# Análise Exploratória — SAEB 2023 (2º ano EF)
**Arquivo:** `TS_ALUNO_2EF.csv`  
**Projeto:** Desigualdade educacional — dados socioeconômicos e desempenho escolar

## 0. Imports e carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.2f}'.format)

COLUNAS_SAEB = [
    'ID_ALUNO', 'ID_REGIAO', 'ID_UF', 'ID_MUNICIPIO', 'ID_ESCOLA',
    'IN_PUBLICA', 'ID_LOCALIZACAO', 'ID_SERIE', 'IN_SITUACAO_CENSO',
    'PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB',
    'IN_ALFABETIZADO', 'IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_MT'
]

df = pd.read_csv('TS_ALUNO_2EF.csv', sep=';', encoding='latin-1', usecols=COLUNAS_SAEB)

# Rótulos
REGIAO_LABELS   = {1: 'Norte', 2: 'Nordeste', 3: 'Sudeste', 4: 'Sul', 5: 'Centro-Oeste'}
PUBLICA_LABELS  = {1: 'Pública', 0: 'Privada'}
LOCAL_LABELS    = {1: 'Urbana', 2: 'Rural'}
ALFAB_LABELS    = {1: 'Alfabetizado', 0: 'Não alfabetizado'}

df['REGIAO']       = df['ID_REGIAO'].map(REGIAO_LABELS)
df['TIPO_ESCOLA']  = df['IN_PUBLICA'].map(PUBLICA_LABELS)
df['LOCALIZACAO']  = df['ID_LOCALIZACAO'].map(LOCAL_LABELS)
df['ALFABETIZADO'] = df['IN_ALFABETIZADO'].map(ALFAB_LABELS)

# Apenas alunos com proficiência válida em ambas as disciplinas
df_valido = df.dropna(subset=['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']).copy()

print(f'Total de alunos: {len(df):,}')
print(f'Com proficiência válida (LP e MT): {len(df_valido):,}')

---
## 1. Estatísticas Descritivas

In [ ]:
print('=== SAEB 2023 — Proficiências (escala SAEB) ===')
display(df_valido[['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']].describe().T.rename(columns={
    'count': 'n', 'mean': 'média', 'std': 'desvio padrão',
    'min': 'mín', '25%': 'Q1', '50%': 'mediana', '75%': 'Q3', 'max': 'máx'
}))

In [ ]:
print('=== Distribuição por Tipo de Escola ===')
display(
    df['TIPO_ESCOLA'].value_counts()
    .to_frame('n')
    .assign(pct=lambda x: (100 * x['n'] / x['n'].sum()).round(1))
)

print('\n=== Distribuição por Região ===')
display(
    df['REGIAO'].value_counts()
    .to_frame('n')
    .assign(pct=lambda x: (100 * x['n'] / x['n'].sum()).round(1))
)

print('\n=== Distribuição por Localização ===')
display(
    df['LOCALIZACAO'].value_counts()
    .to_frame('n')
    .assign(pct=lambda x: (100 * x['n'] / x['n'].sum()).round(1))
)

---
## 2. Distribuições das Proficiências

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label, cor in zip(
    axes,
    ['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB'],
    ['Língua Portuguesa', 'Matemática'],
    ['steelblue', 'tomato']
):
    data = df_valido[col]
    ax.hist(data, bins=50, color=cor, edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(),   color='black',  linestyle='--', linewidth=1.3, label=f'Média: {data.mean():.0f}')
    ax.axvline(data.median(), color='orange', linestyle=':',  linewidth=1.3, label=f'Mediana: {data.median():.0f}')
    ax.set_title(f'Proficiência — {label}')
    ax.set_xlabel('Proficiência SAEB')
    ax.set_ylabel('Frequência')
    ax.legend(fontsize=9)

fig.suptitle('Distribuição das Proficiências — SAEB 2023 (2º EF)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição separando Pública vs Privada
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label in zip(
    axes,
    ['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB'],
    ['Língua Portuguesa', 'Matemática']
):
    sns.kdeplot(
        data=df_valido, x=col, hue='TIPO_ESCOLA',
        ax=ax, fill=True, common_norm=False, alpha=0.4
    )
    ax.set_title(f'{label} — Pública vs Privada')
    ax.set_xlabel('Proficiência SAEB')

fig.suptitle('Distribuição de Proficiência por Tipo de Escola — SAEB 2023', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Correlações Iniciais

In [ ]:
# Correlação LP x Matemática
r = df_valido[['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']].corr().iloc[0, 1]
print(f'Correlação de Pearson — LP × Matemática: r = {r:.3f}')

amostra = df_valido.sample(min(5000, len(df_valido)), random_state=42)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    amostra['PROFICIENCIA_LP_SAEB'], amostra['PROFICIENCIA_MT_SAEB'],
    alpha=0.2, s=10, color='steelblue'
)
ax.set_xlabel('Proficiência — Língua Portuguesa')
ax.set_ylabel('Proficiência — Matemática')
ax.set_title(f'LP × Matemática — SAEB 2023  (r = {r:.2f})')
plt.tight_layout()
plt.show()

In [ ]:
# Proficiência média por Região
prof_regiao = (
    df_valido
    .groupby('REGIAO')[['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']]
    .mean()
    .sort_values('PROFICIENCIA_LP_SAEB', ascending=False)
    .rename(columns={'PROFICIENCIA_LP_SAEB': 'Língua Portuguesa', 'PROFICIENCIA_MT_SAEB': 'Matemática'})
)
print('=== Proficiência média por Região ===')
display(prof_regiao.round(1))

prof_regiao.plot(kind='bar', figsize=(10, 5), rot=15, colormap='Set2')
plt.title('Proficiência Média por Região — SAEB 2023 (2º EF)')
plt.ylabel('Proficiência SAEB')
plt.xlabel('Região')
plt.legend(title='Disciplina')
plt.tight_layout()
plt.show()

In [ ]:
# Proficiência média por Tipo de Escola
prof_escola = (
    df_valido
    .groupby('TIPO_ESCOLA')[['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']]
    .mean()
    .rename(columns={'PROFICIENCIA_LP_SAEB': 'Língua Portuguesa', 'PROFICIENCIA_MT_SAEB': 'Matemática'})
)
print('=== Proficiência média: Pública vs Privada ===')
display(prof_escola.round(1))

# Diferença absoluta
diff = prof_escola.loc['Privada'] - prof_escola.loc['Pública']
print(f'\nDiferença Privada − Pública:')
print(diff.round(1))

In [ ]:
# Proficiência média por Localização (Urbana vs Rural)
prof_local = (
    df_valido
    .groupby('LOCALIZACAO')[['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB']]
    .mean()
    .rename(columns={'PROFICIENCIA_LP_SAEB': 'Língua Portuguesa', 'PROFICIENCIA_MT_SAEB': 'Matemática'})
)
print('=== Proficiência média: Urbana vs Rural ===')
display(prof_local.round(1))

In [ ]:
# Taxa de alfabetização por tipo de escola e região
print('=== Taxa de Alfabetização por Tipo de Escola ===')
alfab_escola = (
    df.groupby('TIPO_ESCOLA')['IN_ALFABETIZADO']
    .mean()
    .mul(100)
    .round(1)
    .to_frame('% Alfabetizados')
)
display(alfab_escola)

print('\n=== Taxa de Alfabetização por Região ===')
alfab_regiao = (
    df.groupby('REGIAO')['IN_ALFABETIZADO']
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame('% Alfabetizados')
)
display(alfab_regiao)